# Chapter 7 — Search In-Depth
## Topic 9: Evaluating Retrieval — Recall@K, MRR, NDCG@K

**Run cells in order.**

- Module 1: Setup — knowledge base + a small labeled evaluation set (graded relevance)
- Module 2: Metrics implemented from scratch (Recall@K, MRR, NDCG@K)
- Module 3: Worked example — verify NDCG@K math by hand against the code
- Module 4: Evaluate every retrieval configuration from Topics 1-7 side by side
- Module 5: Where metrics disagree — a case where Recall@K and NDCG@K tell different stories

**Install:** `pip install sentence-transformers rank-bm25 numpy`


## Module 1: Setup — Knowledge Base + Labeled Evaluation Set

Builds a small labeled evaluation set: for each query, which document IDs
are relevant, AND how relevant (graded 0/1/2) — the ground truth every
metric in this topic depends on.

**Requires:** nothing standalone (loads its own model)


In [ ]:
"""
MODULE 1: Setup -- Knowledge Base + Labeled Evaluation Set

The evaluation set is the actual prerequisite this whole topic depends on.
Each entry: a query, and a dict of {doc_id: graded_relevance} for EVERY
document that has any relevance to that query (0 = irrelevant is implicit
for anything not listed, but is included where useful to show explicitly).
"""

import numpy as np
import math
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

KNOWLEDGE_BASE = [
    {"id": 0, "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "doc_type": "faq"},
    {"id": 1, "text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.", "doc_type": "policy"},
    {"id": 2, "text": "SOP Step 3: Calculate premature withdrawal penalty at 1 percent before processing closure.", "doc_type": "sop"},
    {"id": 3, "text": "Nominee-initiated premature withdrawal due to depositor death is exempt from the standard penalty.", "doc_type": "policy"},
    {"id": 4, "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "doc_type": "product"},
    {"id": 5, "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.", "doc_type": "product"},
    {"id": 6, "text": "At maturity, the FD principal and interest are credited within 2 working days.", "doc_type": "faq"},
]

CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]

# -----------------------------------------------------------------------
# Labeled evaluation set -- ground truth for every metric in this topic.
# Graded relevance: 2 = directly answers the query, 1 = related context,
# 0 or absent = not relevant.
# -----------------------------------------------------------------------
EVAL_SET = [
    {
        "query": "what is the penalty for premature FD withdrawal",
        "relevance": {0: 2, 1: 2, 2: 2, 3: 1},   # 3 near-duplicate direct answers + nominee context
    },
    {
        "query": "senior citizen FD interest rate",
        "relevance": {4: 2},                      # single clear correct answer
    },
    {
        "query": "when do I get my FD money after maturity",
        "relevance": {6: 2, 0: 0},                 # 6 is correct; 0 explicitly marked irrelevant despite topic overlap
    },
    {
        "query": "FD band karna hai penalty kitni hogi",   # Hinglish -- vocabulary mismatch test
        "relevance": {0: 2, 1: 1, 2: 1},
    },
]

print("Loading embedding model (may take a moment on first run)...")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kb_embeddings = model.encode(CORPUS_TEXTS, normalize_embeddings=True, show_progress_bar=False)
for doc in KNOWLEDGE_BASE:
    doc["embedding"] = kb_embeddings[doc["id"]]

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def tokenize(text):
    return text.lower().split()

print(f"\nKnowledge base: {len(KNOWLEDGE_BASE)} chunks")
print(f"Evaluation set: {len(EVAL_SET)} labeled queries\n")
for entry in EVAL_SET:
    print(f"  Query: '{entry['query']}'")
    print(f"    Relevance labels: {entry['relevance']}")

print("\nModule 1 complete. Run Module 2.")


## Module 2: Metrics Implemented From Scratch

Recall@K, Reciprocal Rank (for MRR), and NDCG@K — exactly as specified
in the theory's formulas.

**Requires:** Module 1


In [ ]:
"""
MODULE 2: Metrics From Scratch
"""

def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Fraction of all truly relevant docs that appear in top-K."""
    top_k = set(retrieved_ids[:k])
    if not relevant_ids:
        return 0.0
    return len(top_k & relevant_ids) / len(relevant_ids)


def reciprocal_rank(retrieved_ids: list, relevant_ids: set) -> float:
    """1/rank of the FIRST relevant doc found, 0 if none found."""
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def dcg_at_k(retrieved_ids: list, relevance_grades: dict, k: int) -> float:
    """Discounted Cumulative Gain -- position-weighted graded relevance."""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k], start=1):
        rel = relevance_grades.get(doc_id, 0)
        dcg += rel / math.log2(i + 1)
    return dcg


def ndcg_at_k(retrieved_ids: list, relevance_grades: dict, k: int) -> float:
    """DCG@K normalized by the IDEAL DCG@K (perfect ordering)."""
    actual_dcg = dcg_at_k(retrieved_ids, relevance_grades, k)
    ideal_order = sorted(relevance_grades.keys(), key=lambda d: relevance_grades[d], reverse=True)
    ideal_dcg = dcg_at_k(ideal_order, relevance_grades, k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg


# -----------------------------------------------------------------------
# Quick sanity check on a synthetic example (independent of the real KB)
# -----------------------------------------------------------------------
print("Sanity check -- synthetic example:")
synthetic_retrieved = [10, 20, 30, 40, 50]
synthetic_relevant = {10, 30, 99}   # doc 99 never retrieved -- tests incomplete recall

r_at_3 = recall_at_k(synthetic_retrieved, synthetic_relevant, k=3)
rr = reciprocal_rank(synthetic_retrieved, synthetic_relevant)
print(f"  Retrieved: {synthetic_retrieved}")
print(f"  Relevant (ground truth): {synthetic_relevant}")
print(f"  Recall@3: {r_at_3:.4f}  (expect 2/3 = 0.6667 -- docs 10,30 found in top-3, doc 99 never found)")
print(f"  Reciprocal Rank: {rr:.4f}  (expect 1/1 = 1.0 -- doc 10 is relevant AND at rank 1)")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Worked Example — Verify NDCG@K By Hand

Manually computes the exact worked example from the theory
([2, 0, 1] relevance sequence, NDCG@3 = 0.950) and confirms the code
produces the identical number.

**Requires:** Module 2


In [ ]:
"""
MODULE 3: Worked Example -- Manual NDCG@K Verification

From the theory:
  Retrieved order relevance grades: [2, 0, 1]
  DCG@3  = 2/log2(2) + 0/log2(3) + 1/log2(4) = 2.0 + 0 + 0.5 = 2.5
  Ideal order: [2, 1, 0]
  IDCG@3 = 2/log2(2) + 1/log2(3) + 0/log2(4) = 2.0 + 0.631 + 0 = 2.631
  NDCG@3 = 2.5 / 2.631 = 0.950
"""

# Set up doc IDs matching the [2, 0, 1] relevance sequence in retrieved order
worked_retrieved = ["doc_A", "doc_B", "doc_C"]   # retrieved in this order
worked_relevance = {"doc_A": 2, "doc_B": 0, "doc_C": 1}

# Manual calculation, step by step, matching the theory's worked example
manual_dcg = 2/math.log2(2) + 0/math.log2(3) + 1/math.log2(4)
ideal_order_manual = ["doc_A", "doc_C", "doc_B"]  # sorted by relevance: 2, 1, 0
manual_idcg = 2/math.log2(2) + 1/math.log2(3) + 0/math.log2(4)
manual_ndcg = manual_dcg / manual_idcg

print("Manual calculation (matching the theory's worked example):")
print(f"  DCG@3  = 2/log2(2) + 0/log2(3) + 1/log2(4) = {manual_dcg:.4f}")
print(f"  IDCG@3 = 2/log2(2) + 1/log2(3) + 0/log2(4) = {manual_idcg:.4f}")
print(f"  NDCG@3 = {manual_dcg:.4f} / {manual_idcg:.4f} = {manual_ndcg:.4f}")

# Now via the actual functions from Module 2
code_dcg = dcg_at_k(worked_retrieved, worked_relevance, k=3)
code_ndcg = ndcg_at_k(worked_retrieved, worked_relevance, k=3)

print(f"\nCode calculation (Module 2 functions):")
print(f"  DCG@3  = {code_dcg:.4f}")
print(f"  NDCG@3 = {code_ndcg:.4f}")

assert abs(manual_dcg - code_dcg) < 1e-9, "DCG mismatch!"
assert abs(manual_ndcg - code_ndcg) < 1e-9, "NDCG mismatch!"
print("\n[VERIFIED] Manual calculation matches code output exactly.")
print(f"Expected ~0.950 from the theory: got {code_ndcg:.4f}")

print("\nModule 3 complete. Run Module 4.")


## Module 4: Evaluate Every Retrieval Configuration Side by Side

Runs BM25-only, Dense-only, and Hybrid RRF (Topic 4) against the SAME
labeled evaluation set, computing Recall@K, MRR, and NDCG@K for each --
turning every "should be validated" deferral from Topics 1-8 into an
actual comparison table.

**Requires:** Module 1, Module 2


In [ ]:
"""
MODULE 4: Evaluate Every Retrieval Configuration

Runs BM25-only, Dense-only, and Hybrid RRF against the identical labeled
evaluation set from Module 1, computing all three metrics for each.
"""

# -----------------------------------------------------------------------
# Build the retrievers (mirrors Topics 1, 2, 4)
# -----------------------------------------------------------------------
tokenized_corpus = [tokenize(doc["text"]) for doc in KNOWLEDGE_BASE]
bm25_index = BM25Okapi(tokenized_corpus, k1=1.2, b=0.75)


def bm25_retrieve(query: str, top_k: int = 10) -> list:
    scores = bm25_index.get_scores(tokenize(query))
    scored = [(KNOWLEDGE_BASE[i]["id"], scores[i]) for i in range(len(scores)) if scores[i] > 0]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in scored[:top_k]]


def dense_retrieve(query: str, top_k: int = 10) -> list:
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scored = [(doc["id"], cosine_sim(query_vec, doc["embedding"])) for doc in KNOWLEDGE_BASE]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in scored[:top_k]]


def hybrid_rrf_retrieve(query: str, top_k: int = 10, k_const: int = 60) -> list:
    """Topic 4's RRF, reused here for evaluation."""
    bm25_ranked = bm25_retrieve(query, top_k=10)
    dense_ranked = dense_retrieve(query, top_k=10)

    rrf_scores = {}
    for rank, doc_id in enumerate(bm25_ranked, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k_const + rank)
    for rank, doc_id in enumerate(dense_ranked, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k_const + rank)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in fused[:top_k]]


RETRIEVERS = {
    "BM25-only": bm25_retrieve,
    "Dense-only": dense_retrieve,
    "Hybrid RRF": hybrid_rrf_retrieve,
}

EVAL_K = 3  # matches a realistic top-k passed to generation


def evaluate_retriever(retriever_fn, eval_set: list, k: int) -> dict:
    """Runs a retriever across the full evaluation set, averages all 3 metrics."""
    recalls, rrs, ndcgs = [], [], []
    for entry in eval_set:
        query = entry["query"]
        relevance = entry["relevance"]
        relevant_ids = {doc_id for doc_id, grade in relevance.items() if grade > 0}

        retrieved = retriever_fn(query, top_k=10)

        recalls.append(recall_at_k(retrieved, relevant_ids, k))
        rrs.append(reciprocal_rank(retrieved, relevant_ids))
        ndcgs.append(ndcg_at_k(retrieved, relevance, k))

    return {
        "Recall@{}".format(k): sum(recalls) / len(recalls),
        "MRR": sum(rrs) / len(rrs),
        "NDCG@{}".format(k): sum(ndcgs) / len(ndcgs),
    }


print(f"Evaluating all retrievers at K={EVAL_K} across {len(EVAL_SET)} labeled queries\n")
print(f"{'Configuration':<15} | {'Recall@'+str(EVAL_K):<12} | {'MRR':<8} | {'NDCG@'+str(EVAL_K):<10}")
print("-" * 55)

results_table = {}
for name, retriever_fn in RETRIEVERS.items():
    metrics = evaluate_retriever(retriever_fn, EVAL_SET, k=EVAL_K)
    results_table[name] = metrics
    print(f"{name:<15} | {metrics['Recall@'+str(EVAL_K)]:<12.4f} | {metrics['MRR']:<8.4f} | {metrics['NDCG@'+str(EVAL_K)]:<10.4f}")

print("""
This table is the actual answer to every "should be validated against an
evaluation set" deferral made across Topics 1, 2, and 4. Whichever
configuration wins here, on THIS corpus and THIS query distribution, is
the evidence-based choice -- not the configuration that seemed better by
qualitative inspection alone.
""")

print("Module 4 complete. Run Module 5.")


## Module 5: Where Metrics Disagree

Constructs a specific case where Recall@K and NDCG@K tell genuinely
different stories about the same retrieval result — demonstrating why
Lead-level practice reports all three metrics together, not just one.

**Requires:** Module 1, Module 2


In [ ]:
"""
MODULE 5: Where Metrics Disagree

Two synthetic result orderings that have IDENTICAL Recall@K (same set of
relevant docs found) but different NDCG@K (different quality of ordering
within that set) -- demonstrates why reporting only one metric can hide
a real quality difference.
"""

# Ground truth for this query: doc 100 is the BEST answer (grade 2),
# doc 200 is a secondary, less complete answer (grade 1)
relevance_grades = {100: 2, 200: 1}
relevant_ids = {100, 200}

# Ordering A: best answer first (good ordering)
ordering_A = [100, 200, 300, 400]

# Ordering B: same two relevant docs found, but WORSE answer ranked first
ordering_B = [200, 100, 300, 400]

K = 3

recall_A = recall_at_k(ordering_A, relevant_ids, K)
recall_B = recall_at_k(ordering_B, relevant_ids, K)

ndcg_A = ndcg_at_k(ordering_A, relevance_grades, K)
ndcg_B = ndcg_at_k(ordering_B, relevance_grades, K)

rr_A = reciprocal_rank(ordering_A, relevant_ids)
rr_B = reciprocal_rank(ordering_B, relevant_ids)

print("Two orderings, SAME relevant documents found, DIFFERENT order:\n")
print(f"  Ordering A: {ordering_A}  (best answer, doc 100, ranked FIRST)")
print(f"  Ordering B: {ordering_B}  (best answer, doc 100, ranked SECOND)")

print(f"\n{'Metric':<12} | {'Ordering A':<12} | {'Ordering B':<12} | Same?")
print("-" * 55)
print(f"{'Recall@3':<12} | {recall_A:<12.4f} | {recall_B:<12.4f} | {'YES -- identical' if recall_A == recall_B else 'NO'}")
print(f"{'MRR':<12} | {rr_A:<12.4f} | {rr_B:<12.4f} | {'YES' if rr_A == rr_B else 'NO -- differs'}")
print(f"{'NDCG@3':<12} | {ndcg_A:<12.4f} | {ndcg_B:<12.4f} | {'YES' if ndcg_A == ndcg_B else 'NO -- differs'}")

print("""
Recall@3 is IDENTICAL for both orderings -- both find the same 2 relevant
documents somewhere within the top 3. If Recall@K were the only metric
reported, these two orderings would look equally good.

MRR is ALSO IDENTICAL for both orderings -- MRR only checks whether the
FIRST retrieved document is relevant AT ALL (grade > 0), not how relevant
it is. Since rank 1 in BOTH orderings is a relevant document (100 in A,
200 in B), MRR scores them the same, completely blind to the fact that
doc 100 (grade=2, the BEST answer) is being outranked by doc 200 (grade=1,
a weaker answer) in Ordering B.

NDCG@3 is the ONLY metric of the three that correctly detects Ordering B
is worse -- because it is both position-weighted AND grade-weighted, it
directly penalizes putting the higher-grade document (100) behind the
lower-grade one (200).

This is precisely why the theory insists on reporting all three metrics
together: BOTH Recall@K and MRR missed a genuine quality difference that
only NDCG@K was able to surface. Relying on any single metric -- even
MRR, not just the simpler Recall@K -- can hide real ranking quality
problems that matter for what the generation layer (Chapter 8) actually
receives as context.
""")

print("=" * 70)
print("CHAPTER 7 TOPIC 9 SUMMARY")
print("=" * 70)
print("""
Recall@K: fraction of truly relevant docs found within top-K.
MRR: how quickly the FIRST relevant doc appears, best for single-answer queries.
NDCG@K: position AND grade weighted -- the only metric of the three that
  can distinguish "right documents, wrong order" from genuinely good ranking.

All three were implemented from scratch and VERIFIED against a hand-worked
example (Module 3) before being trusted for real comparisons (Module 4).

Module 4 turned every "should be validated" deferral from Topics 1, 2,
and 4 into an actual side-by-side comparison table on this project's
knowledge base and a labeled evaluation set.

Module 5 proved metrics can disagree -- Recall@K missed a real ordering
quality difference that MRR and NDCG@K both caught. Never rely on a
single metric for a production retrieval decision.

This closes Chapter 7. Every earlier topic's parameter choices (BM25 k1/b,
RRF's k constant, MMR's lambda, whether to rerank, filter selectivity)
can now be validated using the exact methodology built in this topic.

Next: Chapter 8 -- RAG Generation, or Chapter 9 -- Retrieval Putting It
to Work (per the project syllabus).
""")
